Tester mistral

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral",
    temperature=0
)

reponse = llm.invoke(
    "Bonjour, présente-toi en une phrase."
)

print(reponse.content)

 Bonjour, je suis un assistant intelligent conçu pour répondre à vos questions et vous aider dans vos tâches. Comment puis-je vous aider aujourd'hui ?


Importations des bibliothèques

In [1]:
import pandas as pd

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\nibaj\AppData\Local\Temp\ipykernel_38584\2302131627.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Charger le modèle d'embeddings et FAISS

In [10]:
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Trouver automatiquement le dossier docs-rag-agent
dossier_actuel = Path.cwd()

if (dossier_actuel / "data" / "faiss_index").exists():
    BASE_DIR = dossier_actuel
elif (dossier_actuel / "docs-rag-agent" / "data" / "faiss_index").exists():
    BASE_DIR = dossier_actuel / "docs-rag-agent"
elif (dossier_actuel.parent / "data" / "faiss_index").exists():
    BASE_DIR = dossier_actuel.parent
else:
    raise FileNotFoundError(
        f"Index FAISS introuvable depuis : {dossier_actuel}"
    )

INDEX_PATH = BASE_DIR / "data" / "faiss_index"

print("Dossier de travail :", dossier_actuel)
print("Chemin de l'index :", INDEX_PATH)

# Charger le même modèle que pendant la partie 3
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 32}
)

# Charger l'index existant
vectorstore = FAISS.load_local(
    str(INDEX_PATH),
    embeddings=model,
    allow_dangerous_deserialization=True
)

print("Nombre de vecteurs :", vectorstore.index.ntotal)

Dossier de travail : c:\Users\nibaj\Openclassrooms\Projet9_systeme_rag\docs-rag-agent\tests
Chemin de l'index : c:\Users\nibaj\Openclassrooms\Projet9_systeme_rag\docs-rag-agent\data\faiss_index


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Nombre de vecteurs : 38171


Créer la fonctionnde recherches d'évènements

In [3]:
def recuperer_evenements(question, k=5):

    resultats = vectorstore.similarity_search(
        question,
        k=100
    )

    maintenant = pd.Timestamp.now(tz="UTC")

    evenements = []
    ids_vus = set()

    for doc in resultats:

        date_fin = pd.to_datetime(
            doc.metadata.get("date_fin"),
            errors="coerce",
            utc=True
        )

        if pd.isna(date_fin) or date_fin < maintenant:
            continue

        identifiant = doc.metadata.get("id")

        if identifiant in ids_vus:
            continue

        ids_vus.add(identifiant)
        evenements.append(doc)

        if len(evenements) >= k:
            break

    return evenements

Création du prompt Mistral

In [4]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Tu es un assistant culturel pour Puls-Events.

Tu aides les utilisateurs à découvrir des événements à Paris.

Consignes :
- Réponds uniquement à partir des événements fournis.
- N'invente jamais de titre, de lieu ou de date.
- Recommande les événements les plus pertinents.
- Pour chaque événement, indique son titre, son lieu,
  sa date de fin et son URL OpenAgenda.
- Ne présente pas la date de fin comme la date d'une séance.
- Si les informations sont insuffisantes, indique-le.
- Réponds en français, clairement et naturellement.

Événements disponibles :
{contexte}"""
    ),
    ("human", "{question}")
])

Configuration de Mistral et la chaine LangChain

In [5]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

chaine_rag = prompt | llm | StrOutputParser()

J'ai utilisé Mistral en Local via Ollama, comme je n'ai pas la clé API 

Créer la fonnction principal du chatbot

In [6]:
def chatbot_rag(question):

    # 1. Rechercher des événements dans FAISS
    documents = recuperer_evenements(question, k=5)

    if not documents:
        return "Aucun événement correspondant n'a été trouvé."

    # 2. Préparer le contexte pour Mistral
    contexte = ""

    for doc in documents:

        metadata = doc.metadata

        contexte += f"""
Titre : {metadata.get('titre')}
Lieu : {metadata.get('lieu')}
Ville : {metadata.get('ville')}
Date de début : {metadata.get('date_debut')}
Date de fin : {metadata.get('date_fin')}
URL : {metadata.get('url')}
Description : {doc.page_content}

---
"""

    # 3. Envoyer la question et le contexte à Mistral
    reponse = chaine_rag.invoke({
        "question": question,
        "contexte": contexte
    })

    return reponse

Elle interroge FAISS.
Elle récupère jusqu'à cinq événements.
Elle prépare les données de ces événements.
Elle transmet les informations à Mistral.
Elle renvoie la réponse générée.

Tester le chatbot

In [ ]:
from IPython.display import Markdown, display

question = "Quels concerts sont prévus à Paris ?"

reponse = chatbot_rag(question)

display(Markdown(reponse))

 Voici quelques concerts prévus à Paris :

1. BIGA*RANX EN CONCERT AU ZENITH PARIS - LA VILLETTE
   - Titre : BIGA*RANX - ZENITH - PARIS - COMPLET
   - Lieu : Zénith Paris - La Villette
   - Ville : Paris
   - Date de fin : 2026-11-21
   - URL : https://openagenda.com/shotgun/events/bigaranx-zenith-paris-complet

2. K-REEN à la Cigale - Paris
   - Titre : K-REEN à la Cigale - Paris
   - Lieu : La Cigale
   - Ville : Paris
   - Date de fin : 2027-01-08
   - URL : https://openagenda.com/shotgun/events/k-reen-a-la-cigale-paris

3. ISAAC DELUSION - ZENITH - PARIS
   - Titre : ISAAC DELUSION - ZENITH - PARIS
   - Lieu : Zénith Paris - La Villette
   - Ville : Paris
   - Date de fin : 2027-04-24
   - URL : https://openagenda.com/shotgun/events/isaac-delusion-zenith-paris

4. ICE SEALED EYES
   - Titre : ICE SEALED EYES
   - Lieu : La Boule Noire
   - Ville : Paris
   - Date de fin : 2026-11-15
   - URL : https://openagenda.com/shotgun/events/ice-sealed-eyes

5. ÉVEIL DU ROMANTISME | Carolin 

Tester plusieurs réponses à la robustesse du chatbot

In [13]:
questions_test = [
    "Quels concerts sont prévus à Paris ?",
    "Je cherche une exposition artistique à Paris.",
    "Quelles activités peut-on faire avec des enfants ?",
    "Quels événements scientifiques sont prévus à Paris ?",
    "Je cherche un spectacle de théâtre à Paris."
]

for question in questions_test:
    print("\nQUESTION :", question)
    print("-" * 50)

    reponse = chatbot_rag(question)
    print(reponse)

    print("=" * 70)


QUESTION : Quels concerts sont prévus à Paris ?
--------------------------------------------------
 Voici quelques concerts prévus à Paris :

1. BIGA*RANX EN CONCERT AU ZENITH PARIS - LA VILLETTE
   - Titre : BIGA*RANX - ZENITH - PARIS - COMPLET
   - Lieu : Zénith Paris - La Villette
   - Ville : Paris
   - Date de fin : 2026-11-21
   - URL : https://openagenda.com/shotgun/events/bigaranx-zenith-paris-complet

2. K-REEN à la Cigale - Paris
   - Titre : K-REEN à la Cigale - Paris
   - Lieu : La Cigale
   - Ville : Paris
   - Date de fin : 2027-01-08
   - URL : https://openagenda.com/shotgun/events/k-reen-a-la-cigale-paris

3. ISAAC DELUSION - ZENITH - PARIS
   - Titre : ISAAC DELUSION - ZENITH - PARIS
   - Lieu : Zénith Paris - La Villette
   - Ville : Paris
   - Date de fin : 2027-04-24
   - URL : https://openagenda.com/shotgun/events/isaac-delusion-zenith-paris

4. ICE SEALED EYES
   - Titre : ICE SEALED EYES
   - Lieu : La Boule Noire
   - Ville : Paris
   - Date de fin : 2026-11-15

Creer le tableau d'évaluation

In [14]:
import pandas as pd

questions_evaluation = [
    "Quels concerts sont prévus à Paris ?",
    "Je cherche une exposition artistique à Paris.",
    "Quelles activités peut-on faire avec des enfants ?",
    "Quels événements scientifiques sont prévus à Paris ?",
    "Je cherche un spectacle de théâtre à Paris."
]

df_evaluation = pd.DataFrame({
    "question": questions_evaluation,
    "reponse_ia": ["" for _ in questions_evaluation],
    "reponse_reference": ["" for _ in questions_evaluation],
    "evaluation": ["" for _ in questions_evaluation],
    "commentaire": ["" for _ in questions_evaluation]
})

display(df_evaluation)

,question,reponse_ia,reponse_reference,evaluation,commentaire
0,Quels concerts sont prévus à Paris ?,,,,
1,Je cherche une exposition artistique à Paris.,,,,
2,Quelles activités peut-on faire avec des enfan...,,,,
3,Quels événements scientifiques sont prévus à P...,,,,
4,Je cherche un spectacle de théâtre à Paris.,,,,


Récuperer les réponses sans tronquer

In [15]:
from IPython.display import Markdown, display

question = questions_evaluation[1]

documents = recuperer_evenements(question, k=5)

print("Événements récupérés par FAISS :")

for doc in documents:
    print(doc.metadata["titre"])
    print(doc.metadata["date_fin"])
    print(doc.metadata["url"])
    print("-" * 40)

print("\nRéponse de Mistral :")
display(Markdown(chatbot_rag(question)))

Événements récupérés par FAISS :
PhotoSaintGermain-Marek Piasecki, A Friday Person
2027-01-30 18:00:00+00:00
https://openagenda.com/bicentenaire-photographie-fr/events/galerie-le-minotaure
----------------------------------------
PhotoSaintGermain-Guillaume Bihan et Daria Svertilova, Clément Chapillon, Daphné Nan Le Sergent
2026-11-28 18:00:00+00:00
https://openagenda.com/bicentenaire-photographie-fr/events/galerie-du-crous-x-cnap
----------------------------------------
Des photographies aux archives : pour quoi faire ?
2026-12-23 16:30:00+00:00
https://openagenda.com/bicentenaire-photographie-fr/events/des-photographies-aux-archives-pour-quels-usages
----------------------------------------
Exposition "Traverser le monde : L'Exposition universelle de 1867 et la photographie"
2027-02-22 16:30:00+00:00
https://openagenda.com/bicentenaire-photographie-fr/events/exposition-traverser-le-monde-lexposition-universelle-de-1867-a-paris-et-la-photographie
--------------------------------------

 Je suis heureux de vous aider à trouver une exposition artistique à Paris. Voici quelques événements qui pourraient vous intéresser :

1. Titre : PhotoSaintGermain-Marek Piasecki, A Friday Person
   Lieu : Galerie Le Minotaure
   Ville : Paris
   Date de fin : 2027-01-30 18:00:00+00:00
   URL : https://openagenda.com/bicentenaire-photographie-fr/events/galerie-le-minotaure

2. Titre : PhotoSaintGermain-Guillaume Bihan et Daria Svertilova, Clément Chapillon, Daphné Nan Le Sergent
   Lieu : Galerie du Crous
   Ville : Paris
   Date de fin : 2026-11-28 18:00:00+00:00
   URL : https://openagenda.com/bicentenaire-photographie-fr/events/galerie-du-crous-x-cnap

3. Titre : Des photographies aux archives : pour quoi faire ?
   Lieu : Archives de Paris
   Ville : Paris
   Date de fin : 2026-12-23 16:30:00+00:00
   URL : https://openagenda.com/bicentenaire-photographie-fr/events/des-photographies-aux-archives-pour-quels-usages

4. Titre : Exposition "Traverser le monde : L'Exposition universelle de 1867 et la photographie"
   Lieu : Archives nationales - Site de Paris
   Ville : Paris
   Date de fin : 2027-02-22 16:30:00+00:00
   URL : https://openagenda.com/bicentenaire-photographie-fr/events/exposition-traverser-le-monde-lexposition-universelle-de-1867-a-paris-et-la-photographie

5. Titre : Exposition The Dreamers 2026
   Lieu : Galerie La Moulinette
   Ville : Paris
   Date de fin : 2026-12-22 18:00:00+00:00
   URL : https://openagenda.com/bicentenaire-photographie-fr/events/exposition-the-dreamers-2026

Je vous souhaite une bonne visite à ces expositions et j'espère que vous trouverez ce qui vous convient le mieux.

Comparaison des réponses aux documents récupérés

In [16]:
question = "Je cherche une exposition artistique à Paris."

documents = recuperer_evenements(question, k=5)

for i, doc in enumerate(documents, 1):
    print(f"ÉVÉNEMENT {i}")
    print("Titre :", doc.metadata["titre"])
    print("Lieu :", doc.metadata["lieu"])
    print("Date de fin :", doc.metadata["date_fin"])
    print("URL :", doc.metadata["url"])
    print("-" * 60)

ÉVÉNEMENT 1
Titre : PhotoSaintGermain-Marek Piasecki, A Friday Person
Lieu : Galerie Le Minotaure
Date de fin : 2027-01-30 18:00:00+00:00
URL : https://openagenda.com/bicentenaire-photographie-fr/events/galerie-le-minotaure
------------------------------------------------------------
ÉVÉNEMENT 2
Titre : PhotoSaintGermain-Guillaume Bihan et Daria Svertilova, Clément Chapillon, Daphné Nan Le Sergent
Lieu : Galerie du Crous
Date de fin : 2026-11-28 18:00:00+00:00
URL : https://openagenda.com/bicentenaire-photographie-fr/events/galerie-du-crous-x-cnap
------------------------------------------------------------
ÉVÉNEMENT 3
Titre : Des photographies aux archives : pour quoi faire ?
Lieu : Archives de Paris
Date de fin : 2026-12-23 16:30:00+00:00
URL : https://openagenda.com/bicentenaire-photographie-fr/events/des-photographies-aux-archives-pour-quels-usages
------------------------------------------------------------
ÉVÉNEMENT 4
Titre : Exposition "Traverser le monde : L'Exposition universe

Compléter le tableau d'évaluation

In [17]:
df_evaluation.loc[1, "reponse_ia"] = reponse
df_evaluation.loc[1, "reponse_reference"] = (
    "Cinq événements artistiques à Paris : "
    "PhotoSaintGermain - Marek Piasecki, "
    "PhotoSaintGermain - Galerie du Crous, "
    "Des photographies aux archives, "
    "Traverser le monde et The Dreamers 2026."
)
df_evaluation.loc[1, "evaluation"] = "Correcte"
df_evaluation.loc[1, "commentaire"] = (
    "Les cinq événements correspondent aux documents FAISS. "
    "Titres, lieux, dates de fin et liens conformes."
)

display(df_evaluation[["question", "evaluation", "commentaire"]])

,question,evaluation,commentaire
0,Quels concerts sont prévus à Paris ?,,
1,Je cherche une exposition artistique à Paris.,Correcte,Les cinq événements correspondent aux document...
2,Quelles activités peut-on faire avec des enfan...,,
3,Quels événements scientifiques sont prévus à P...,,
4,Je cherche un spectacle de théâtre à Paris.,,


In [18]:
resultats = df_evaluation["evaluation"].value_counts()
print(resultats)

taux_correct = (
    (df_evaluation["evaluation"] == "Correcte").mean() * 100
)

print(f"Taux de réponses correctes : {taux_correct:.1f} %")

evaluation
            4
Correcte    1
Name: count, dtype: int64
Taux de réponses correctes : 20.0 %


Corriger le calcul

In [19]:
# Conserver uniquement les réponses déjà évaluées
df_notes = df_evaluation[
    df_evaluation["evaluation"].isin([
        "Correcte",
        "Partiellement correcte",
        "Incorrecte"
    ])
]

# Calculer le taux sur les réponses évaluées
taux_correct = (
    (df_notes["evaluation"] == "Correcte").mean() * 100
)

print("Réponses évaluées :", len(df_notes))
print("Réponses restantes :", len(df_evaluation) - len(df_notes))
print(f"Taux de réponses correctes : {taux_correct:.1f} %")

Réponses évaluées : 1
Réponses restantes : 4
Taux de réponses correctes : 100.0 %
